In [5]:
import re

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import spacy
import pandas as pd
import matplotlib.pyplot as plt

In [10]:
# Load data
bbc_data = pd.read_csv("Files/bbc_news.csv")
bbc_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Unnamed: 0   1000 non-null   int64 
 1   index        1000 non-null   int64 
 2   title        1000 non-null   object
 3   pubDate      1000 non-null   object
 4   guid         1000 non-null   object
 5   link         1000 non-null   object
 6   description  1000 non-null   object
dtypes: int64(2), object(5)
memory usage: 54.8+ KB


In [16]:
titles = pd.DataFrame(bbc_data["title"]) # Series
titles.head()

,title
0,Can I refuse to work?
1,'Liz Truss the Brief?' World reacts to UK poli...
2,Rationing energy is nothing new for off-grid c...
3,The hunt for superyachts of sanctioned Russian...
4,Platinum Jubilee: 70 years of the Queen in 70 ...


### Clean Data

In [17]:
# Lowercase
titles["lowercase"] = titles['title'].str.lower()
titles.head()

,title,lowercase
0,Can I refuse to work?,can i refuse to work?
1,'Liz Truss the Brief?' World reacts to UK poli...,'liz truss the brief?' world reacts to uk poli...
2,Rationing energy is nothing new for off-grid c...,rationing energy is nothing new for off-grid c...
3,The hunt for superyachts of sanctioned Russian...,the hunt for superyachts of sanctioned russian...
4,Platinum Jubilee: 70 years of the Queen in 70 ...,platinum jubilee: 70 years of the queen in 70 ...


In [19]:
# Remove stop words
en_stopwords = stopwords.words("english")
titles["no_stopwords"] = titles.lowercase.apply(lambda x: " ".join(word for word in x.split() if word not in en_stopwords))
titles["no_stopwords"]

0                                           refuse work?
1      'liz truss brief?' world reacts uk political t...
2        rationing energy nothing new off-grid community
3          hunt superyachts sanctioned russian oligarchs
4            platinum jubilee: 70 years queen 70 seconds
                             ...                        
995    dominic raab: third senior civil servant gives...
996                  highlights: radacanu beats uytvanck
997         pictures: mountain bikers descend snowy peak
998    companies must help cut living costs, says new...
999       beware online car sale scams, consumers warned
Name: no_stopwords, Length: 1000, dtype: object

In [20]:
# Remove punctuations
titles["no_punc"] = titles.apply(lambda x: re.sub(r"[^\w\s]", "", x.no_stopwords), axis=1)
titles.no_punc

0                                            refuse work
1      liz truss brief world reacts uk political turmoil
2         rationing energy nothing new offgrid community
3          hunt superyachts sanctioned russian oligarchs
4             platinum jubilee 70 years queen 70 seconds
                             ...                        
995    dominic raab third senior civil servant gives ...
996                   highlights radacanu beats uytvanck
997          pictures mountain bikers descend snowy peak
998    companies must help cut living costs says new ...
999        beware online car sale scams consumers warned
Name: no_punc, Length: 1000, dtype: object

In [22]:
# Tokenize
titles["raw_tokens"] = titles["title"].apply(lambda x: word_tokenize(x))
titles["clean_tokens"] = titles["no_punc"].apply(lambda x: word_tokenize(x))
titles[["raw_tokens", "clean_tokens"]]

,raw_tokens,clean_tokens
0,"[Can, I, refuse, to, work, ?]","[refuse, work]"
1,"['Liz, Truss, the, Brief, ?, ', World, reacts,...","[liz, truss, brief, world, reacts, uk, politic..."
2,"[Rationing, energy, is, nothing, new, for, off...","[rationing, energy, nothing, new, offgrid, com..."
3,"[The, hunt, for, superyachts, of, sanctioned, ...","[hunt, superyachts, sanctioned, russian, oliga..."
4,"[Platinum, Jubilee, :, 70, years, of, the, Que...","[platinum, jubilee, 70, years, queen, 70, seco..."
...,...,...
995,"[Dominic, Raab, :, Third, senior, civil, serva...","[dominic, raab, third, senior, civil, servant,..."
996,"[Highlights, :, Radacanu, beats, Uytvanck]","[highlights, radacanu, beats, uytvanck]"
997,"[In, pictures, :, Mountain, bikers, descend, s...","[pictures, mountain, bikers, descend, snowy, p..."
998,"[Companies, must, help, cut, living, costs, ,,...","[companies, must, help, cut, living, costs, sa..."


In [24]:
# Lemmatization
lemmatizer = WordNetLemmatizer()
titles["clean_lemmatized"] = titles["clean_tokens"].apply(lambda tokens: [lemmatizer.lemmatize(token) for token in tokens])
titles[["clean_lemmatized", "clean_tokens"]]

,clean_lemmatized,clean_tokens
0,"[refuse, work]","[refuse, work]"
1,"[liz, truss, brief, world, reacts, uk, politic...","[liz, truss, brief, world, reacts, uk, politic..."
2,"[rationing, energy, nothing, new, offgrid, com...","[rationing, energy, nothing, new, offgrid, com..."
3,"[hunt, superyachts, sanctioned, russian, oliga...","[hunt, superyachts, sanctioned, russian, oliga..."
4,"[platinum, jubilee, 70, year, queen, 70, second]","[platinum, jubilee, 70, years, queen, 70, seco..."
...,...,...
995,"[dominic, raab, third, senior, civil, servant,...","[dominic, raab, third, senior, civil, servant,..."
996,"[highlight, radacanu, beat, uytvanck]","[highlights, radacanu, beats, uytvanck]"
997,"[picture, mountain, bikers, descend, snowy, peak]","[pictures, mountain, bikers, descend, snowy, p..."
998,"[company, must, help, cut, living, cost, say, ...","[companies, must, help, cut, living, costs, sa..."


In [26]:
tokens_raw_list = sum(titles.raw_tokens, [])
tokens_clean_list = sum(titles.clean_lemmatized, [])